# PhiState L4 — Training and Evaluation Journal

**Your saved model, one notebook.** Mount Drive, load the pinned code, audit the actual data and token exposure, inspect results, and use the controls below to start or resume an independent broad-data continuation.

**Run all is audit-only. Training starts when you press Start / Resume.** A GPU is required for training; reading reports and native CPU chat do not require one. In Colab select **Runtime → Change runtime type → GPU** when training.

Latest user-reported results (these are recorded history, not results produced by this notebook):

| Experiment | Observed result |
|---|---|
| Repaired pretraining | Step 65,000; best validation mean 5.91241 BPT; generations remained incoherent |
| Conversation training | 2,000 updates; answer BPT 6.18734 → 5.18160; repetitive generic replies |
| Tiny learning check | 16/16 exact training answers at update 400; two of four reworded questions answered correctly |
| Generalization curriculum | Final definitions 7/12; six trained code functions 6/6; arithmetic 0/24; macro score 0.39583 |
| C++ inference | Earlier parity checks passed; latest curriculum export was withheld by its quality gate |

The model can learn and shows limited wording transfer. It is **not yet a reliable general chatbot**. The curriculum checkpoint is kept as an experiment; it is not the source for optional continuation.


In [ ]:
# 1. Mount your existing Drive. This does not start training.
from pathlib import Path
import os, sys, json, subprocess, importlib.util

ROOT = Path('/content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
assert ROOT.is_dir(), f'Existing training folder not found: {ROOT}'
print('Saved project:', ROOT)


In [ ]:
# 2. Load an exact code revision in a separate worktree.
PIN = '284b8d86f6ce8d296e033b7ac2eb1a2dbda33c13'
BRANCH = 'feat/phistate-native-cpp-trial'
REPO = Path('/content/phistate_l4/quantoniumoslinux')
CODE = Path('/content/phistate-journal-' + PIN[:12])

def command(args):
    subprocess.run([str(x) for x in args], check=True)

if not REPO.exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    command(['git', 'clone', '--branch', BRANCH,
             'https://github.com/Anharmoniclabs/quantoniumoslinux.git', REPO])
command(['git', '-C', REPO, 'fetch', 'origin', BRANCH])
if not CODE.exists():
    command(['git', '-C', REPO, 'worktree', 'add', '--detach', CODE, PIN])
actual = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == PIN, 'Worktree revision mismatch'
command(['git', '-C', CODE, 'diff', '--exit-code', 'HEAD'])

missing = [name for name in ['sentencepiece', 'pyarrow', 'pandas', 'matplotlib', 'ipywidgets']
           if importlib.util.find_spec(name) is None]
if missing:
    command([sys.executable, '-m', 'pip', 'install', '-q', *missing])
LAB = CODE / 'runtime/experiments/phi_runtime_lab'
# Avoid reusing a different revision already imported by an older notebook cell.
for name in ['journal_control', 'chat_recovery_train', 'chat_recovery_data',
             'chat_recovery_cpp', 'moe_train', 'moe_model', 'phistate',
             'tiny_learning_check', 'native_cpp_trial']:
    module = sys.modules.get(name)
    if module and Path(getattr(module, '__file__', '')).parent != LAB:
        del sys.modules[name]
sys.path.insert(0, str(LAB))
import torch
import pandas as pd
from IPython.display import display, clear_output
import journal_control as journal
print('Pinned code:', PIN)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none; audit / CPU chat available')


## Audit the saved work

The next cells read your actual files. Checkpoint counters overlap across parent and child runs: **do not add their totals together**. The 60K checkpoint used a different tokenization history, so dividing all historical exposure by the repaired corpus size would not give a meaningful epoch count.


In [ ]:
# 3. Checkpoint inventory. Avoid loading a checkpoint while its trainer is saving.
try:
    CHECKPOINTS = journal.inventory(ROOT)
    display(pd.DataFrame(CHECKPOINTS))
except RuntimeError as error:
    CHECKPOINTS = []
    print(error)
    print('A worker may be active. Use Refresh below, then rerun inventory after it stops.')


In [ ]:
# 4. Actual prepared dataset counts, tokens and multi-turn examples.
AUDIT = journal.audit_data(ROOT)
display(pd.DataFrame([{'split': k, 'tokens': v} for k, v in AUDIT['bins'].items()]))
display(pd.DataFrame(AUDIT['sft']))
for note in AUDIT['notes']:
    print('•', note)


In [ ]:
# 5. Sampling by example is different from sampling by token.
# This estimates input-token exposure, not gradient weights or actual observed counts.
train_stats = [r for r in AUDIT['sft'] if r['split'] == 'train']
mix = []
for r in train_stats:
    micro_share = .8 * r['configured_sft_example_share']
    mix.append({'source': r['source'], 'microbatch_share': micro_share,
                'expected_input_tokens_per_microbatch': micro_share * r['mean_input_tokens']})
for domain, domain_share in [('original_code', .55), ('original_general', .45)]:
    mix.append({'source': domain, 'microbatch_share': .2 * domain_share,
                'expected_input_tokens_per_microbatch': .2 * domain_share * 128})
total = sum(r['expected_input_tokens_per_microbatch'] for r in mix)
for r in mix:
    r['expected_input_token_share'] = r['expected_input_tokens_per_microbatch'] / total
MIX = pd.DataFrame(mix)
display(MIX)
print('The configured 20% corpus replay is 20% of microbatches, not necessarily 20% of input tokens.')
print('Each microbatch loss is averaged separately; these token shares are not loss weights.')


## Optional broad-data continuation

This is a **bounded continuation experiment using the existing prepared data**, not a newly expanded corpus or a proven coherence fix. It resumes the broad conversational checkpoint, excluding the 16-example and 279-prompt diagnostic curricula.

The first Start / Resume copies the conversation `latest.pt` and `best.pt` into `broad_continuation_v1`. It preserves weights, Adafactor state and sampling state. Existing pretraining, chat, and diagnostic checkpoints stay intact.

The existing recipe is unchanged: 80% supervised conversational microbatches (OpenAssistant/Python/support/math), 20% original-corpus replay. The default adds at most **2,000 SFT updates** to the copied checkpoint. On your reported state this targets SFT step 4,000. There is no promise that this budget will produce coherent chat.

Held-out answer BPT, per-source loss, replay validation and actual generated responses remain visible. This runner selects best by answer BPT subject to a retention check; **best does not mean coherent**. It stops for the existing retention/plateau rules. Native export is manual after reviewing results.

Start / Resume runs in the background so Refresh and Save-and-Stop remain usable. A hard Colab runtime deletion can lose updates after the last saved checkpoint; saves are normally every 100 updates and at evaluations/stops. A new runtime resumes from Drive.


In [ ]:
# 6. Choose the fixed budget BEFORE the first launch. Running this cell only creates controls.
ADDITIONAL_SFT_STEPS = 2000 # @param {type:'integer'}
import ipywidgets as widgets

start_button = widgets.Button(description='Start / Resume', button_style='success')
stop_button = widgets.Button(description='Save-and-Stop', button_style='warning')
refresh_button = widgets.Button(description='Refresh')
control_output = widgets.Output()

def on_start(_):
    with control_output:
        clear_output(wait=True)
        try:
            journal.launch(ROOT, LAB, ADDITIONAL_SFT_STEPS)
        except Exception as error:
            print(type(error).__name__ + ':', error)

def on_stop(_):
    with control_output:
        journal.request_stop(ROOT)

def on_refresh(_):
    with control_output:
        clear_output(wait=True)
        journal.monitor(ROOT)

start_button.on_click(on_start)
stop_button.on_click(on_stop)
refresh_button.on_click(on_refresh)
display(widgets.HBox([start_button, stop_button, refresh_button]), control_output)
print('No training starts until you press Start / Resume.')


In [ ]:
# 7. Standalone progress check. Rerun whenever you want.
journal.monitor(ROOT)


## Review loss and generations together

The conversation and its fork use the same prepared validation set and settings. Their answer BPT values can be plotted together. The curriculum pass rates and pretraining BPT measure different things and must not be placed on the same loss curve.


In [ ]:
# 8. Plot compatible conversation validation curves when files exist.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
found = False
for name, folder in [('conversation', ROOT / 'chat_recovery_v1/run'),
                     ('broad continuation', ROOT / 'broad_continuation_v1')]:
    path = folder / 'metrics.jsonl'
    points = []
    if path.exists():
        with path.open() as stream:
            for line in stream:
                try:
                    r = json.loads(line)
                    points.append((r['sft_step'], r['val_answer_bpt']))
                except (json.JSONDecodeError, KeyError):
                    continue
    if points:
        ax.plot([p[0] for p in points], [p[1] for p in points], marker='o', label=name)
        found = True
ax.set(xlabel='SFT update', ylabel='Held-out answer BPT (lower is better)',
       title='Conversation validation — inspect generated answers too')
ax.grid(alpha=.25)
if found: ax.legend()
else: print('No conversation metrics files yet.')
plt.show()


In [ ]:
# 9. Show latest saved generations from each run, without loading models.
for label, folder in [('conversation', ROOT / 'chat_recovery_v1/run'),
                      ('broad continuation', ROOT / 'broad_continuation_v1')]:
    paths = sorted(folder.glob('generation_*.json')) if folder.exists() else []
    print('\nRUN:', label)
    if not paths:
        print('No saved generation samples yet.')
        continue
    path = paths[-1]
    print('SAMPLES:', path.name)
    for sample in json.loads(path.read_text()):
        print('\nUSER:', sample['prompt'])
        print('ASSISTANT:', repr(sample['answer']))


In [ ]:
# 10. Display actual final reports from the diagnostic runs.
for label, relative in [('tiny learning', 'tiny_learning_check_v1/report.json'),
                        ('generalization', 'generalization_v1/report.json')]:
    path = ROOT / relative
    print('\nREPORT:', label)
    if not path.exists():
        print('Not present in this Drive folder.')
        continue
    r = json.loads(path.read_text())
    if label == 'tiny learning':
        e = r.get('training_evaluation', {})
        print({k:e.get(k) for k in ['exact_answers', 'prompt_wins', 'training_answer_bpt']})
        for p in r.get('unseen_wordings', []):
            print(p['question'], '→', repr(p['generated']))
    else:
        print('Selected update:', r.get('selected_phase_step'))
        print(json.dumps(r.get('final_evaluation', {}).get('buckets', {}), indent=2))
        print('Native export eligible:', r.get('native_export_eligible'))
        for p in r.get('open_questions', []):
            print(p['question'], '→', repr(p['generated']))


## Test a saved checkpoint directly

The preview below uses PyTorch and the same tokenizer/template. It does not train or replace anything. The separate C++ section uses a verified native export. For comparisons, keep the question and decoding settings fixed. An appropriate greeting alone does not demonstrate general question answering.


In [ ]:
# 11. Optional single-prompt checkpoint preview. Tick RUN_PREVIEW, then run this cell.
RUN_PREVIEW = False # @param {type:'boolean'}
PREVIEW_RUN = 'conversation' # @param ['conversation', 'generalization', 'broad continuation']
QUESTION = 'What is the difference between RAM and storage?' # @param {type:'string'}
if RUN_PREVIEW:
    import gc
    from chat_recovery_train import run_lock
    from moe_train import ckpt_load
    from chat_recovery_data import sha
    from tiny_learning_check import decode
    import sentencepiece as spm
    folders = {'conversation': ROOT / 'chat_recovery_v1/run',
               'generalization': ROOT / 'generalization_v1',
               'broad continuation': ROOT / 'broad_continuation_v1'}
    folder = folders[PREVIEW_RUN]
    assert (folder / 'best.pt').is_file(), 'Selected best checkpoint is not present.'
    assert not journal.worker_active(ROOT / 'broad_continuation_v1'), 'Stop the background worker before GPU preview.'
    with run_lock(folder):
        model, checkpoint = ckpt_load(str(folder / 'best.pt'))
        tokenizer_path = ROOT / 'corpus_repaired_v1/tokenizer.model'
        assert checkpoint.get('tokenizer_sha256') == sha(tokenizer_path), 'Tokenizer mismatch'
        tokenizer = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
        from chat_recovery_data import prompt_text
        assert len(tokenizer.encode(prompt_text([{'role':'user','content':QUESTION}]))) + 128 <= 512, 'Shorten the prompt.'
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model.to(device).eval()
        print('Checkpoint:', folder / 'best.pt', '| step:', checkpoint['step'], '| backend:', device)
        answer, ended = decode(model, tokenizer, QUESTION, 128)
        print('ANSWER:', answer)
        print('WHITESPACE VISIBLE:', repr(answer), '| EOS:', ended)
        del model, checkpoint
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('Preview is off. Tick RUN_PREVIEW to test the selected checkpoint.')


## Native C++ inference

The existing conversation native export remains available. The curriculum's failed export gate stays in force. A broad continuation can be exported manually after reviewing its generations; parity means C++ matches PyTorch, not that answers are correct.


In [ ]:
# 12. Optional broad-candidate export after the worker has finished.
EXPORT_BROAD_CANDIDATE = False # @param {type:'boolean'}
if EXPORT_BROAD_CANDIDATE:
    from chat_recovery_cpp import export
    folder = ROOT / 'broad_continuation_v1'
    assert not journal.worker_active(folder), 'Wait for the worker to finish or save-and-stop.'
    progress = json.loads((folder / 'progress.json').read_text())
    assert progress['status'] in ['complete', 'early_stopped', 'retention_stopped', 'stopped'], 'Run has not reached a saved stop.'
    export(folder, ROOT / 'chat_recovery_v1/data')
    print('Separate C++ candidate exported. Review correctness; existing conversation export is unchanged.')
else:
    print('Export is off. Use the existing verified conversation model below, or review a new candidate first.')


In [ ]:
# 13. Interactive C++ chat. Tick RUN_CHAT and run THIS cell. Blank input exits.
RUN_CHAT = False # @param {type:'boolean'}
NATIVE_RUN = 'conversation' # @param ['conversation', 'generalization', 'broad continuation']
if RUN_CHAT:
    from chat_recovery_cpp import chat
    folders = {'conversation': ROOT / 'chat_recovery_v1/run',
               'generalization': ROOT / 'generalization_v1',
               'broad continuation': ROOT / 'broad_continuation_v1'}
    folder = folders[NATIVE_RUN]
    if (folder / 'native_latest.json').is_file():
        chat(folder)
    else:
        print('No verified native export for this selection. The generalization run did not pass its export gate.')
else:
    print('Chat is off. Tick RUN_CHAT to enter the interactive prompt. Do not paste notebook code into YOU.')


## Save this session's audit

This creates a timestamped JSON journal entry beside your runs. It records counts and saved progress, not model weights. Checkpoints remain in their own run folders. Do not use the single composite score as a claim of general intelligence or coding ability.


In [ ]:
# 14. Save a compact audit snapshot to Drive.
from datetime import datetime, timezone
from chat_recovery_train import atomic_json
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
journal_dir = ROOT / 'journal'
journal_dir.mkdir(exist_ok=True)
progress = {}
for label, folder in [('conversation', ROOT / 'chat_recovery_v1/run'),
                      ('generalization', ROOT / 'generalization_v1'),
                      ('broad continuation', ROOT / 'broad_continuation_v1')]:
    path = folder / 'progress.json'
    if path.exists(): progress[label] = json.loads(path.read_text())
snapshot = journal_dir / ('audit_' + stamp + '.json')
atomic_json(snapshot, {'code_commit': PIN, 'utc_time': stamp, 'data_audit': AUDIT,
                      'checkpoint_inventory': CHECKPOINTS, 'saved_progress': progress})
print('Saved:', snapshot)


## What to send back

After a training phase, share the latest generation samples, per-source held-out answer BPT, replay validation, and `progress.json`. If only a fixed set of answers improves while unfamiliar questions remain mismatched, report that as limited transfer. Do not silently promote the candidate.

For terminal monitoring, the continuation log is:

```bash
tail -f /content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/broad_continuation_v1/training.log
```

Ctrl+C exits `tail` only. Use **Save-and-Stop** in this notebook to request a saved training stop. Wait for `status: stopped` before disconnecting when possible.
